In [5]:
import pandas as pd

# Load all datasets with low_memory=False to suppress warnings
fourth_downs_2021 = pd.read_csv('fourth_downs_2018_2019.csv', low_memory=False)
fourth_downs_2022 = pd.read_csv('fourth_downs_2020_2021.csv', low_memory=False)
fourth_downs_2023 = pd.read_csv('fourth_downs_2022_2023.csv', low_memory=False)
fourth_downs_2024 = pd.read_csv('fourth_downs_2024_2025.csv', low_memory=False)
play_callers = pd.read_excel('playcallers_by_week.xlsx', engine='openpyxl')
hard_count = pd.read_csv('hard_count_attempts.csv', low_memory=False)

# Step 1: Stack all fourth down datasets vertically
all_fourth_downs = pd.concat([
    fourth_downs_2021,
    fourth_downs_2022,
    fourth_downs_2023,
    fourth_downs_2024
], ignore_index=True)

print(f"Total fourth down plays: {len(all_fourth_downs)}")

# Step 2: Add flags to identify play types
all_fourth_downs['is_fourth_down'] = 1
all_fourth_downs['is_hard_count'] = 0

hard_count['is_fourth_down'] = 0
hard_count['is_hard_count'] = 1

# Step 3: Combine fourth downs and hard count plays
# Use play_id + game_id as the unique identifier
all_plays = pd.concat([all_fourth_downs, hard_count], ignore_index=True)

# Mark plays that appear in both datasets
play_key = all_plays['play_id'].astype(str) + '_' + all_plays['game_id'].astype(str)
duplicate_plays = play_key.duplicated(keep=False)

# For duplicates, keep one record but mark both flags as True
all_plays_dedup = all_plays.groupby(['play_id', 'game_id'], as_index=False).agg({
    'is_fourth_down': 'max',  # Will be 1 if any record has it
    'is_hard_count': 'max',   # Will be 1 if any record has it
    **{col: 'first' for col in all_plays.columns if col not in ['is_fourth_down', 'is_hard_count', 'play_id', 'game_id']}
})

print(f"Total unique plays (fourth downs + hard count): {len(all_plays_dedup)}")
print(f"Plays that are BOTH fourth down AND hard count: {(all_plays_dedup['is_fourth_down'] == 1) & (all_plays_dedup['is_hard_count'] == 1).sum()}")

# Step 4: Join with play callers - filter for offensive play callers
offensive_play_callers = play_callers[play_callers['side_of_ball'] == 'off'].copy()

# Join on team (posteam) and season
final_dataset = all_plays_dedup.merge(
    offensive_play_callers[['team', 'season', 'playcaller', 'weeks']],
    left_on=['posteam', 'season'],
    right_on=['team', 'season'],
    how='left'
)

# Clean up duplicate team column
final_dataset = final_dataset.drop(columns=['team'])

print(f"\nFinal dataset shape: {final_dataset.shape}")
print(f"Plays with play caller info: {final_dataset['playcaller'].notna().sum()}")
print(f"Plays missing play caller info: {final_dataset['playcaller'].isna().sum()}")

# Show summary
print("\n--- Summary Statistics ---")
print(f"Fourth down plays only: {(final_dataset['is_fourth_down'] == 1) & (final_dataset['is_hard_count'] == 0).sum()}")
print(f"Hard count plays only: {(final_dataset['is_fourth_down'] == 0) & (final_dataset['is_hard_count'] == 1).sum()}")
print(f"Both fourth down AND hard count: {((final_dataset['is_fourth_down'] == 1) & (final_dataset['is_hard_count'] == 1)).sum()}")

# Save the final dataset
final_dataset.to_csv('combined_fourth_downs_hardcount.csv', index=False)
print("\n✓ Saved to 'combined_fourth_downs_hardcount.csv'")

Total fourth down plays: 33444


/tmp/ipykernel_23812/3699970519.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  all_fourth_downs['is_fourth_down'] = 1
/tmp/ipykernel_23812/3699970519.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  all_fourth_downs['is_hard_count'] = 0
/tmp/ipykernel_23812/3699970519.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, u

Total unique plays (fourth downs + hard count): 35195
Plays that are BOTH fourth down AND hard count: 0        False
1        False
2        False
3         True
4         True
         ...  
35190     True
35191     True
35192     True
35193     True
35194     True
Name: is_fourth_down, Length: 35195, dtype: bool

Final dataset shape: (38762, 376)
Plays with play caller info: 35549
Plays missing play caller info: 3213

--- Summary Statistics ---
Fourth down plays only: 0        False
1        False
2        False
3         True
4         True
         ...  
38757     True
38758     True
38759     True
38760     True
38761     True
Name: is_fourth_down, Length: 38762, dtype: bool
Hard count plays only: 0         True
1         True
2         True
3        False
4        False
         ...  
38757    False
38758    False
38759    False
38760    False
38761    False
Name: is_fourth_down, Length: 38762, dtype: bool
Both fourth down AND hard count: 327

✓ Saved to 'combined_fourth_downs_ha